# Sanity Check for DRR Renderers
This notebook loads a single CT volume and renders it across all 5 engines (Plastimatch, Monte Carlo, DVR, DiffDRR, DeepDRR) for visual inspection.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import matplotlib.pyplot as plt
from pathlib import Path

from utils.logger import setup_logger, get_logger
from data import load_ct_volume, make_diffdrr_subject
from renderers.config import DEFAULT_GEOMETRY
from renderers import (
    build_dvr_renderer,
    build_diffdrr_renderer,
    generate_plastimatch_drr,
    generate_mc_drr,
    generate_deepdrr_drr,
)

setup_logger()
logger = get_logger("sanity_check")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 1. Load a single CT volume
target_size = 128
res = 200 # DRR resolution

logger.info("Loading CT volume...")
volume_tensor, subject, voxel_spacing = load_ct_volume(target_size=target_size)

if subject is None:
    subject = make_diffdrr_subject(volume_tensor, voxel_spacing)

vol = volume_tensor.to(device)
images = {}

In [ ]:
# 2. Render from all engines
logger.info("1. Rendering with Plastimatch (Ground Truth)...")
img_plastimatch = generate_plastimatch_drr(volume_tensor, voxel_spacing, res, device)
if img_plastimatch is not None:
    images["Plastimatch"] = img_plastimatch.cpu().squeeze().numpy()
else:
    logger.warning("Plastimatch rendered None.")

logger.info("2. Rendering with Monte Carlo...")
img_mc = generate_mc_drr(volume_tensor, voxel_spacing, res, device)
if img_mc is not None:
    images["Monte Carlo"] = img_mc.cpu().squeeze().numpy()
else:
    logger.warning("Monte Carlo rendered None.")
    
logger.info("3. Rendering with DVR...")
renderer, cameras = build_dvr_renderer(res, 320, target_size, voxel_spacing, device)
if renderer is not None:
    with torch.no_grad():
        img_dvr = renderer(vol, cameras, norm_type="normalized")
        images["DVR"] = img_dvr.cpu().squeeze().numpy()
else:
    logger.warning("DVR renderer could not be built.")
        
logger.info("4. Rendering with DiffDRR (Siddon)...")
drr_siddon, rot, xyz = build_diffdrr_renderer(res, target_size, subject, voxel_spacing, device)
if drr_siddon is not None:
    with torch.no_grad():
        img_siddon = drr_siddon(rot, xyz, parameterization="euler_angles", convention="ZXY")
        images["DiffDRR\n(Siddon)"] = img_siddon.cpu().squeeze().numpy()
else:
    logger.warning("DiffDRR renderer could not be built.")
        
try:
    from diffdrr.drr import DRR as DiffDRR_module
    logger.info("5. Rendering with DiffDRR (Trilinear)...")
    delx = target_size * voxel_spacing / res
    drr_tri = DiffDRR_module(
        subject,
        sdd=DEFAULT_GEOMETRY.sdd,
        height=res,
        delx=delx,
        renderer="trilinear",
    ).to(device)
    with torch.no_grad():
        img_trilinear = drr_tri(rot, xyz, parameterization="euler_angles", convention="ZXY")
        # Flip horizontally to match standard coordinate orientation
        img_trilinear = torch.flip(img_trilinear, dims=[-1])
        images["DiffDRR\n(Trilinear)"] = img_trilinear.cpu().squeeze().numpy()
except Exception as exc:
    logger.debug(f"Skipping DiffDRR Trilinear: {exc}")
    
logger.info("6. Rendering with DeepDRR...")
img_deepdrr = generate_deepdrr_drr(volume_tensor, voxel_spacing, res, device)
if img_deepdrr is not None:
    images["DeepDRR"] = img_deepdrr.cpu().squeeze().numpy()
else:
    logger.warning("DeepDRR rendered None.")

In [ ]:
# 3. Plotting results side-by-side
n_images = len(images)
if n_images == 0:
    logger.error("No images were rendered! Cannot create plot.")
else:
    logger.info(f"Plotting {n_images} images for visual inspection...")
    fig, axes = plt.subplots(1, n_images, figsize=(4 * n_images, 4))
    if n_images == 1:
        axes = [axes]
        
    for ax, (title, img) in zip(axes, images.items()):
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=14, pad=10)
        ax.axis("off")
        
    plt.tight_layout()
    plt.show()